# Hierarchical Equal Risk Contribution

Raffinot (2018) cuts the tree of the assets into clusters using the shape of the dendrogram, then gives each cluster an equal share of risk, following the Equal Risk Contribution approach. This notebook computes HERC weights, checks the risk shares of the two sectors, and compares HERC, HRP and equal weights on held-out returns.

In [ ]:
Pollis.packages("PortfolioOptimisers", "Clarabel", "Plots")

In [ ]:
using PortfolioOptimisers, Clarabel, Random, Statistics, Plots

## The data

1,000 trading days of daily simple returns for 8 assets in two sectors. A market factor moves every asset, a sector factor moves each sector, and Student-t shocks with 5 degrees of freedom give fat tails. The true annualised means are 12% for Tech and 6% for Util, so the estimates can be checked against them.

In [ ]:
Random.seed!(42)
T, N = 1_000, 8
t5(n) = sqrt(3 / 5) .* randn(n) ./ sqrt.(vec(sum(abs2, randn(5, n); dims = 1)) ./ 5)   # unit-variance Student-t(5) shocks
mkt = 0.008 .* t5(T); tech = 0.010 .* t5(T); util = 0.005 .* t5(T)   # market and sector factors
mu_d = [fill(0.12, 4); fill(0.06, 4)] ./ 252   # true daily means
R = hcat([j <= 4 ? mu_d[j] .+ 1.2 .* mkt .+ tech .+ 0.012 .* t5(T) : mu_d[j] .+ 0.6 .* mkt .+ util .+ 0.006 .* t5(T) for j in 1:N]...)
asset_names = ["Tech1", "Tech2", "Tech3", "Tech4", "Util1", "Util2", "Util3", "Util4"]

rd = ReturnsResult(; nx = asset_names, X = R)
rf_daily = 0.02 / 252
slv = Solver(; name = :clarabel, solver = Clarabel.Optimizer, settings = Dict("verbose" => false),
             check_sol = (; allow_local = true, allow_almost = true))
show_weights(w) = foreach((nm, wi) -> println(rpad(nm, 8), lpad(round(100 * wi; digits = 2), 8), " %"), asset_names, w)

## HERC and HRP weights

In [ ]:
res = optimise(HierarchicalEqualRiskContribution(; ri = Variance(), ro = Variance()), rd)
w_herc, pr = res.w, res.pr
w_hrp = optimise(HierarchicalRiskParity(; r = Variance()), rd).w
show_weights(w_herc)
bar(asset_names, 100 .* w_herc; label = "HERC", ylabel = "Weight (%)", xrotation = 45)
scatter!(asset_names, 100 .* w_hrp; label = "HRP", ms = 6)

## Risk shares of the sectors

The Euler contributions add up to the portfolio's variance, so they can be summed by sector.

In [ ]:
for (nm, w) in [("HERC", w_herc), ("HRP", w_hrp)]
    rc = risk_contribution(Variance(), w, pr)
    s = rc ./ sum(rc)
    println(rpad(nm, 6), "Tech ", round(100 * sum(s[1:4]); digits = 1), " % of risk   Util ", round(100 * sum(s[5:8]); digits = 1), " % of risk")
end

## On held-out returns

Each method is refitted on 504 days and held for the next 63, on the same folds. The standard error of each Sharpe ratio follows Lo (2002) for iid returns.

In [ ]:
cv = IndexWalkForward(504, 63)
for (nm, s) in [("HERC", HierarchicalEqualRiskContribution()), ("HRP", HierarchicalRiskParity()), ("Equal Weight", EqualWeighted())]
    x = cross_val_predict(s, rd, cv).mrd.X
    sr = (mean(x) - rf_daily) / std(x)
    println(rpad(nm, 14), "Sharpe ", lpad(round(sqrt(252) * sr; digits = 2), 6), "   SE ", round(sqrt(252 * (1 + sr^2 / 2) / length(x)); digits = 2),
            "   vol ", round(100 * sqrt(252) * std(x); digits = 1), " %")
end

## Exercises

1. Give the Util sector six assets and the Tech sector two. Does HERC still give the two sectors similar shares of risk? Does HRP?
2. Replace `Variance()` with `ConditionalValueatRisk(; alpha = 0.05)` for `ri` and `ro`. How do the weights change?
3. Are the Sharpe ratios on held-out returns more than two standard errors apart?